# Structured Output & Function Calling

**Track:** LLMOps  
**Toolchain:** OpenAI Function Calling · Instructor · Pydantic  
**Objective:** Learn how to make LLMs return reliably structured data that your code can parse — the bridge between natural language and deterministic software.

---

## 🧠 The #1 Problem with LLMs in Production

LLMs return **free-form text.** Your code needs **structured data.** This mismatch causes 80% of production LLM bugs.

```python
# What you WANT from the LLM:
{"sentiment": "positive", "confidence": 0.92, "topics": ["pricing", "quality"]}

# What the LLM ACTUALLY returns:
"The sentiment of this review is mostly positive, with a confidence of around 92%.
 The main topics discussed are pricing and quality."

# Now try doing json.loads() on that 😱
```

### The Reliability Spectrum

| Method | Reliability | How It Works |
|--------|-----------|-------------|
| Ask nicely ("return JSON") | ~50% | LLM sometimes adds extra text |
| JSON mode (`response_format`) | ~95% | Forces valid JSON, but no schema guarantee |
| Function calling / tool_use | ~99% | LLM fills pre-defined function arguments |
| Structured Output (JSON schema) | ~99.9% | LLM output guaranteed to match schema |
| Instructor + retry | ~99.99% | Validates output, auto-retries on failure |

---

## ✅ Knowledge Check
### Q1: Why use JSON mode or function calling instead of regex to parse outputs?
<details><summary>Answer</summary>
LLMs can natively output structured data with schemas, drastically reducing parsing errors and enabling complex nested structures that regex struggles with.
</details>
### Q2: How does function calling interact with the LLM's natural language response?
<details><summary>Answer</summary>
The LLM returns an intelligent 'call function' intent sequence instead of direct text, allowing the application to execute logic and return the result to the LLM to formulate the final answer.
</details>


## 🔨 Practical Practice
### Exercise 1: Schema Definition
Define a JSON schema using Pydantic for a weather API response and write the corresponding function calling specification.
### Exercise 2: Function Loop
Implement a basic function calling loop that allows the model to query an external dummy database for product prices.


## 📑 Table of Contents

* [🧠 The #1 Problem with LLMs in Production](#the-1-problem-with-llms-in-production)
  * [The Reliability Spectrum](#the-reliability-spectrum)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Function Calling: Let the LLM Use Your Tools](#1-function-calling-let-the-llm-use-your-tools)
  * [🤔 What is Function Calling?](#what-is-function-calling)
  * [Function Calling vs Tool Use (Same Thing, Different Names)](#function-calling-vs-tool-use-same-thing-different-names)
* [2 · Pydantic + Instructor: The Production Pattern](#2-pydantic-instructor-the-production-pattern)
  * [🤔 What is Instructor?](#what-is-instructor)
* [3 · Multi-Tool Orchestration](#3-multi-tool-orchestration)
  * [🤔 What Happens When the LLM Needs Multiple Tools?](#what-happens-when-the-llm-needs-multiple-tools)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [🧠 Key Rules](#key-rules)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Parsing raw text outputs from an LLM (`json.loads(response)`) is the #1 cause of production pipeline crashes. Seniors FORCE the LLM to output guaranteed schemas using Function Calling and Pydantic validators.

**Mental Model:** Instead of asking an artist to randomly draw a chart, Function Calling is handing the artist a strict architectural blueprint (JSON schema) and forcing them to fill in the exact blanks.

**Common Junior Pitfall:** Using generic "Return JSON" instructions and praying it works. Always use libraries like Instructor or native API `tool_choice` to guarantee structure, combined with automated retry-loops for safety.

---


In [ ]:
# Cell 1 — Install
!pip install -q pydantic openai

## 1 · Function Calling: Let the LLM Use Your Tools

### 🤔 What is Function Calling?

Instead of asking the LLM to write JSON, you tell it: "Here are functions you can call. Pick the right one and fill in the arguments."

**Analogy:** Instead of asking someone to write you a form, you give them a blank form with labeled fields to fill in.

```
You define:    get_weather(city: str, unit: str = "celsius")
User says:     "What's the weather like in Paris?"
LLM returns:   {"function": "get_weather", "arguments": {"city": "Paris", "unit": "celsius"}}
Your code:     Actually calls get_weather("Paris", "celsius")
```

### Function Calling vs Tool Use (Same Thing, Different Names)

| Provider | What They Call It | API Field |
|----------|------------------|----------|
| OpenAI | Function calling → Tools | `tools` |
| Anthropic | Tool use | `tools` |
| Google | Function calling | `tools` |
| Open source | Depends on framework | Varies |

In [ ]:
# Cell 2 — Function calling flow simulation
import json

# Step 1: Define your functions (tools)
tool_definitions = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current stock price for a ticker symbol.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "Stock ticker symbol (e.g., AAPL, GOOGL)"},
                    "currency": {"type": "string", "enum": ["USD", "EUR", "GBP"], "default": "USD"}
                },
                "required": ["ticker"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_documents",
            "description": "Search internal knowledge base for relevant documents.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"},
                    "department": {"type": "string", "enum": ["engineering", "finance", "hr", "legal"]},
                    "max_results": {"type": "integer", "default": 5}
                },
                "required": ["query"]
            }
        }
    },
]

# Step 2: Simulate LLM choosing functions
def simulate_function_calling(user_message, tools):
    """Simulate what the LLM returns when given function definitions."""
    # In production, the LLM itself decides which function to call
    if 'stock' in user_message.lower() or 'price' in user_message.lower():
        return {
            'function': 'get_stock_price',
            'arguments': {'ticker': 'AAPL', 'currency': 'USD'}
        }
    elif 'document' in user_message.lower() or 'policy' in user_message.lower():
        return {
            'function': 'search_documents',
            'arguments': {'query': user_message, 'department': 'hr', 'max_results': 3}
        }
    return None

# Step 3: Execute the function
def execute_function(call):
    """Execute the function the LLM chose."""
    if call['function'] == 'get_stock_price':
        return {'price': 198.50, 'change': '+1.2%', 'currency': call['arguments']['currency']}
    elif call['function'] == 'search_documents':
        return {'results': ['Vacation policy v2.1', 'Remote work guidelines'], 'count': 2}
    return None

# Demo
queries = [
    "What's Apple's stock price?",
    "Find our vacation policy document",
]

print('🔧 Function Calling Flow')
print('=' * 60)
for q in queries:
    print(f'\n  User: "{q}"')
    call = simulate_function_calling(q, tool_definitions)
    print(f'  LLM chose: {call["function"]}({json.dumps(call["arguments"])})')
    result = execute_function(call)
    print(f'  Function result: {json.dumps(result)}')
    print(f'  LLM final response: Generates natural language using the result')

print(f'\n💡 The LLM NEVER runs code. It only CHOOSES which function to call')
print(f'   and FILLS IN the arguments. Your code does the actual execution.')

---

## 2 · Pydantic + Instructor: The Production Pattern

### 🤔 What is Instructor?

Instructor is a library that wraps LLM APIs and **guarantees** the output matches your Pydantic model. If the LLM returns invalid data, Instructor automatically RETRIES with an error description.

```
Without Instructor:  LLM → "some text" → json.loads() → KeyError → crash 💥
With Instructor:     LLM → Pydantic validates → retry if invalid → guaranteed schema ✅
```

In [ ]:
# Cell 3 — Instructor pattern (simulated)
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional
from enum import Enum
import json, random

# Define EXACTLY what you want the LLM to return
class Sentiment(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"
    MIXED = "mixed"

class ProductReview(BaseModel):
    """Structured extraction from a product review."""
    product_name: str = Field(description="Name of the product mentioned")
    sentiment: Sentiment = Field(description="Overall sentiment of the review")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence 0-1")
    pros: List[str] = Field(default_factory=list, description="Positive aspects")
    cons: List[str] = Field(default_factory=list, description="Negative aspects")
    price_mentioned: Optional[float] = Field(None, description="Price if mentioned")
    would_recommend: bool = Field(description="Would the reviewer recommend?")

class InstructorSimulator:
    """Simulates the Instructor library's validate-and-retry pattern."""
    
    def __init__(self, model_cls, max_retries=3):
        self.model_cls = model_cls
        self.max_retries = max_retries
    
    def extract(self, llm_raw_output):
        """Try to parse LLM output into the Pydantic model, retry on failure."""
        for attempt in range(self.max_retries):
            try:
                data = json.loads(llm_raw_output)
                result = self.model_cls(**data)
                return result, attempt + 1
            except (json.JSONDecodeError, ValidationError) as e:
                print(f'    ⚠️ Attempt {attempt+1} failed: {str(e)[:80]}')
                # In real Instructor: send error back to LLM, ask it to fix
                # Simulate: fix the output
                llm_raw_output = self._simulate_retry(llm_raw_output)
        
        raise ValueError(f'Failed after {self.max_retries} attempts')
    
    def _simulate_retry(self, bad_output):
        """Simulate LLM fixing its output."""
        return json.dumps({
            'product_name': 'MacBook Pro', 'sentiment': 'positive',
            'confidence': 0.92, 'pros': ['performance', 'display'],
            'cons': ['price'], 'price_mentioned': 2499.99, 'would_recommend': True
        })

instructor = InstructorSimulator(ProductReview)

# Test 1: Valid output
print('📋 Structured Output Extraction')
print('=' * 60)

valid_output = json.dumps({
    'product_name': 'iPhone 16', 'sentiment': 'positive',
    'confidence': 0.88, 'pros': ['camera', 'battery life'],
    'cons': ['no USB-C cable included'], 'price_mentioned': 999.0,
    'would_recommend': True
})
print(f'\n  Test 1: Valid JSON')
result, attempts = instructor.extract(valid_output)
print(f'    ✅ Parsed in {attempts} attempt: {result.product_name}, {result.sentiment.value}')

# Test 2: Invalid output (bad confidence value)
print(f'\n  Test 2: Invalid JSON (needs retry)')
invalid_output = '{not valid json at all}'
result, attempts = instructor.extract(invalid_output)
print(f'    ✅ Parsed after {attempts} attempts: {result.product_name}')

print(f'\n💡 Instructor auto-retries with the validation error as feedback.')
print(f'   In production, this gives you 99.99% structured output reliability.')

---

## 3 · Multi-Tool Orchestration

### 🤔 What Happens When the LLM Needs Multiple Tools?

Real applications need the LLM to call MULTIPLE tools in sequence or parallel:

```
User: "Compare AAPL and GOOGL stock prices and find related news"

LLM decides:
  1. Call get_stock_price("AAPL")  ← parallel
  2. Call get_stock_price("GOOGL") ← parallel
  3. Call search_news("AAPL GOOGL stock")  ← after prices
  4. Generate comparison using all results
```

In [ ]:
# Cell 4 — Multi-tool orchestration
import json, time

class ToolOrchestrator:
    """Manages multi-tool LLM interactions with cost tracking."""
    
    def __init__(self):
        self.tools = {}
        self.call_log = []
        self.total_cost = 0.0
    
    def register(self, name, func, cost_per_call=0.001):
        self.tools[name] = {'func': func, 'cost': cost_per_call}
    
    def execute_plan(self, plan):
        """Execute a series of tool calls decided by the LLM."""
        results = {}
        for step in plan:
            tool_name = step['tool']
            args = step['args']
            
            tool = self.tools.get(tool_name)
            if not tool:
                results[step['id']] = {'error': f'Unknown tool: {tool_name}'}
                continue
            
            # Check dependencies
            if 'depends_on' in step:
                dep_result = results.get(step['depends_on'])
                if dep_result and 'error' not in dep_result:
                    args['context'] = dep_result
            
            result = tool['func'](**args)
            self.total_cost += tool['cost']
            results[step['id']] = result
            self.call_log.append({'tool': tool_name, 'args': args, 'result': result})
        
        return results

# Register tools
orch = ToolOrchestrator()
orch.register('get_price', lambda ticker, **kw: {'ticker': ticker, 'price': round(100 + hash(ticker) % 200, 2)}, cost_per_call=0.0001)
orch.register('search_news', lambda query, **kw: {'articles': [f'News about {query[:20]}...'], 'count': 1}, cost_per_call=0.001)
orch.register('calculate', lambda expression, **kw: {'result': eval(expression)}, cost_per_call=0.0)

# LLM creates this execution plan
plan = [
    {'id': 'price_aapl', 'tool': 'get_price', 'args': {'ticker': 'AAPL'}},
    {'id': 'price_googl', 'tool': 'get_price', 'args': {'ticker': 'GOOGL'}},
    {'id': 'diff', 'tool': 'calculate', 'args': {'expression': '198.50 - 175.30'}, 'depends_on': 'price_aapl'},
    {'id': 'news', 'tool': 'search_news', 'args': {'query': 'AAPL vs GOOGL'}},
]

print('🔧 Multi-Tool Orchestration')
print('=' * 60)
results = orch.execute_plan(plan)
for step_id, result in results.items():
    print(f'  {step_id}: {json.dumps(result)}')
print(f'\n  Total cost: ${orch.total_cost:.4f} | Tools called: {len(orch.call_log)}')

print(f'\n💡 The LLM plans which tools to call and in what order.')
print(f'   Your orchestrator handles execution, dependencies, and cost tracking.')

---

## 🎯 Summary & Key Takeaways

| Technique | Reliability | Use When |
|-----------|-----------|----------|
| Prompt-based ("return JSON") | ~50% | Quick prototypes only |
| JSON mode | ~95% | Simple structures |
| Function calling | ~99% | LLM needs to use your tools |
| Pydantic + Instructor | ~99.99% | Production structured extraction |

### 🧠 Key Rules

1. **Never trust raw LLM output** — always validate with Pydantic
2. **Use function calling** when the LLM needs to interact with external systems
3. **Always implement retries** — even the best models fail occasionally
4. **Track costs** — each tool call adds latency and potentially cost

**Next →** `20_llmops_context_engineering.ipynb` — Module 6 — LLMOps & Context Engineering: The Agentic Frontier